# Exploratory Data Analysis
---

In [2]:
import pandas as pd

df = pd.read_csv("../data/clean/Telco_Cus_Churn_Clean.csv")

## Univariate Analysis

In [3]:
print("--- DISTRIBUSI CHURN ---")
churn_counts = df['Churn'].value_counts()
churn_percents = df['Churn'].value_counts(normalize=True) * 100

dist_churn = pd.DataFrame({'Jumlah': churn_counts, 'Percentage': churn_percents})
dist_churn

--- DISTRIBUSI CHURN ---


,Jumlah,Percentage
Churn,,
No,5174,73.463013
Yes,1869,26.536987


In [4]:
print("\n--- SUMMARY OF TENURE & MONTHLY CHARGES STATISTICS ---")
df[['tenure', 'MonthlyCharges']].describe()


--- SUMMARY OF TENURE & MONTHLY CHARGES STATISTICS ---


,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [10]:
print("--- OUTLIER DETECTION USING IQR ---")

for col in ['tenure', 'MonthlyCharges', 'TotalCharges'] :
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q1 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Total Outlier in Column {col} : {len(outliers)} from {len(df)} rows")

--- OUTLIER DETECTION USING IQR ---
Total Outlier in Column tenure : 0 from 7043 rows
Total Outlier in Column MonthlyCharges : 13 from 7043 rows
Total Outlier in Column TotalCharges : 940 from 7043 rows


## Bivariate Analysis

In [5]:
print("\n--- ANALYSIS CHURN VS CONTRACT ---")
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.round(2)


--- ANALYSIS CHURN VS CONTRACT ---


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


In [6]:
print("\n--- ANALySIS CHURN VS INTERNET SERVICE ---")
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
internet_churn.round(2)


--- ANALySIS CHURN VS INTERNET SERVICE ---


Churn,No,Yes
InternetService,,
DSL,81.04,18.96
Fiber optic,58.11,41.89
No,92.60,7.40


In [7]:
print("\n--- ANALYSIS CHURN VS PAYMENT METHOD ---")
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.round(2)


--- ANALYSIS CHURN VS PAYMENT METHOD ---


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.29,16.71
Credit card (automatic),84.76,15.24
Electronic check,54.71,45.29
Mailed check,80.89,19.11


In [11]:
from scipy.stats import chi2_contingency

print("--- CHI-SQUARE SIGNIFICANCE TEST FOR CHURN ---")
target_features = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

for feature in target_features:
    contingency_table = pd.crosstab(df[feature], df['Churn'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # If the p-value < 0.05, then the variable has a significant effect
    status = "Significant" if p < 0.05 else "Not Significant"
    print(f"Churn Relationship vs {feature} : p-value = {p:.5f} ({status})")

--- CHI-SQUARE SIGNIFICANCE TEST FOR CHURN ---
Churn Relationship vs Contract : p-value = 0.00000 (Significant)
Churn Relationship vs InternetService : p-value = 0.00000 (Significant)
Churn Relationship vs PaymentMethod : p-value = 0.00000 (Significant)
Churn Relationship vs gender : p-value = 0.48658 (Not Significant)


In [13]:
from scipy.stats import ttest_ind

print("\n--- T-TEST: DIFFERENCE IN AVERAGE MONTHLY CHARGES ---")
churn_yes = df[df['Churn'] == "Yes"]['MonthlyCharges']
churn_no = df[df['Churn'] == "No"]['MonthlyCharges']

stat, p_val = ttest_ind(churn_yes, churn_no, equal_var=False)
status_ttest = "Significant" if p_val < 0.05 else "Not Significant"
print(f"Difference of Monthly Charges against Churn: p-value = {p_val:.5f} ({status_ttest})")


--- T-TEST: DIFFERENCE IN AVERAGE MONTHLY CHARGES ---
Difference of Monthly Charges against Churn: p-value = 0.00000 (Significant)


## Multivariate Analysis

In [8]:
print("\n--- Correlation Matrix of Numerical Variables ---")
matrix_corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()
matrix_corr.round(2)


--- Correlation Matrix of Numerical Variables ---


,tenure,MonthlyCharges,TotalCharges
tenure,1.00,0.25,0.83
MonthlyCharges,0.25,1.00,0.65
TotalCharges,0.83,0.65,1.00


In [9]:
print("\n--- SEGMENT ANALYSIS: AVERAGE TENURE & COST BY CHURN STATUS ---")
segmentaion = df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean()
segmentaion.round(2)


--- SEGMENT ANALYSIS: AVERAGE TENURE & COST BY CHURN STATUS ---


,tenure,MonthlyCharges,TotalCharges
Churn,,,
No,37.57,61.27,2549.91
Yes,17.98,74.44,1531.80


In [14]:

print("--- ADVANCE MULTIVARIATE ANALYSIS: GROUP TENURE SEGMENTATION & INTERNET VS CHURN ---")

# Create new column to group tenure ( in years )
df['tenure_group'] = pd.cut(df['tenure'],
                            bins=[0, 12, 24, 48, 60, 72],
                            labels=['0-1 Tahun', '1-2 Tahun', '2-4 Tahun', '4-5 Tahun','>5 Tahun'])

# Create a Pivot Table to see the Churn percentage == 'Yes'
pivot_advance = df.pivot_table(index='InternetService',
                               columns='tenure_group',
                               values='Churn',
                               aggfunc=lambda x: (x == "Yes").sum() / len(x) * 100)

pivot_advance.round(2)

--- ADVANCE MULTIVARIATE ANALYSIS: GROUP TENURE SEGMENTATION & INTERNET VS CHURN ---


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2552\2556114844.py:9: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_advance = df.pivot_table(index='InternetService',


tenure_group,0-1 Tahun,1-2 Tahun,2-4 Tahun,4-5 Tahun,>5 Tahun
InternetService,,,,,
DSL,40.27,19.64,10.79,7.02,2.78
Fiber optic,69.88,48.78,36.62,24.61,12.42
No,18.15,3.77,1.78,3.11,0.36
